# Quick start, GPU: the same scene on the MuJoCo Warp simulator

The twin of `quickstart_cpu_mujoco.ipynb`: the same arm, task and control. Moving to the GPU is three fields of `.sim()`: `simulator="warp"`,
a CUDA `device`, and the number of envs. The robot is compiled exactly as for the CPU simulator, so the tensors live on the device and
thousands of envs step at once.

Needs an NVIDIA GPU and the extras: `pip install -e ".[env,robosuite,warp]"` (`mujoco` and `mujoco-warp` of one release).
Reference: `docs/simulators.md` and `docs/configs/sim.md` (`WarpOptions`).

In [1]:
import logging
import time

import torch
import warp as wp

from robotdesigner.config import BaseEnvConfig, Chain, FixedGains, WarpOptions
from robotdesigner.tasks.robosuite import LiftObject, RobosuiteParams

import robosuite  # noqa: E402,F401  its import-time warnings about optional extras are harmless
logging.getLogger("robosuite_logs").setLevel(logging.WARNING)   # set after the import: robosuite resets its logger level on import
if hasattr(wp, "LOG_WARNING"):
    wp.config.log_level = wp.LOG_WARNING   # no banner and no per-kernel compile lines in the output (Warp >= 1.18)
else:
    wp.config.quiet = True                  # the same switch in older Warp releases

assert torch.cuda.is_available(), "this notebook needs a CUDA device"
DEVICE = "cuda:0"
print(torch.cuda.get_device_name(DEVICE))

[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /work/yn030245/venv/robotdesigner/lib/python3.13/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)


NVIDIA H100


## The config, with the builder

The same chain as the CPU quick start, with 64 envs. Then one more `.sim()` call selects the GPU simulator; the options of the simulator
you switch to become its defaults (`WarpOptions.default()`), because the options of one simulator cannot be kept for another.

In [2]:
lift = (
    BaseEnvConfig()
    .robot(Chain(n=3, axes="zyy", total_length=0.9, total_mass=3.0))          # a yaw joint, then two pitch joints; the rest is the base's
    .control(mode="joint_pos_abs", gains=FixedGains(kp=60.0, kd=6.0))
    .task(params=RobosuiteParams(name="Lift", env_kwargs={}, objective=LiftObject("cube", height=0.84),
                                 reset_noise=0.0, terminate_on_success=False),
          max_episode_steps=50)                                                # the mount is the provider's default for Lift
    .sim(num_envs=64)
)

In [3]:
gpu = lift.sim(simulator="warp", device=DEVICE)
print(gpu.resolve().sim.options)                                   # njmax / nconmax per world, the CUDA graph on

env = gpu.build()
print(f"robot_hash={env.robot_hash} env_hash={env.env_hash} device={env.device} batch_size={tuple(env.batch_size)}")
print("same robot as the CPU config:", env.robot_hash == lift.resolve().robot.config_hash)

WarpOptions(njmax=128, nconmax=32, cuda_graph=True)
robot_hash=287b31f43ec4 env_hash=9c49a64cf4a5 device=cuda:0 batch_size=(64,)
same robot as the CPU config: True


`WarpOptions` sizes the per-world buffers. `njmax` (constraints) and `nconmax` (contacts) are capacities, not tuning knobs: a scene that needs
more raises `CapacityError` instead of silently dropping contacts. `cuda_graph=True`, the default, captures the physics sub-steps of a step as one CUDA
graph and replays them as a single launch. To give your own options, pass them in the same call:
`lift.sim(simulator="warp", device=DEVICE, options=WarpOptions(njmax=256, nconmax=64, cuda_graph=True))`.

## Roll out a random policy

Same keys and shapes as the CPU simulator; the tensors are on the GPU.

In [4]:
traj = env.rollout(50)
print("rollout batch shape:", tuple(traj.shape), "| device:", traj.device)
print("task observations:", sorted(traj["next", "task"].keys()))
print("final cube z in the robot's base frame (m), first 4 envs:", traj["next", "task", "cube_pos"][:4, -1, 2].tolist())

rollout batch shape: (64, 50) | device: cuda:0
task observations: ['cube_pos', 'cube_quat', 'ee_pos']
final cube z in the robot's base frame (m), first 4 envs: [-0.09090256690979004, -0.09090256690979004, -0.09090256690979004, -0.09090256690979004]


## How fast, on this GPU

Steady-state `env.step` after a warm-up, with the device synchronised before each clock read (the first call includes kernel compilation and graph
capture). Each size runs with the CUDA graph (the default) and without it, which steps kernel by kernel. Without the graph the step is bound by kernel-launch
cost, so it also depends on how busy the CPU is. `benchmarks/bench_warp.py` does this properly, per setting and with CPU baselines; this cell is a quick look.
Numbers depend on the GPU, the scene and the CPU.

In [5]:
def env_steps_per_s(num_envs: int, cuda_graph: bool, steps: int = 50, warmup: int = 10) -> float:
    options = WarpOptions(njmax=128, nconmax=32, cuda_graph=cuda_graph)         # the defaults, with the graph on or off
    e = gpu.sim(num_envs=num_envs, options=options).task(max_episode_steps=10_000).build()     # a variant is another builder call
    td = e.reset()
    for i in range(warmup + steps):
        if i == warmup:
            torch.cuda.synchronize(); t0 = time.perf_counter()
        td["action"] = e.action_spec.rand()
        td = e.step(td)["next"]
    torch.cuda.synchronize()
    rate = num_envs * steps / (time.perf_counter() - t0)
    e.close()
    return rate

for n in (64, 1024, 4096):
    graph, eager = env_steps_per_s(n, True), env_steps_per_s(n, False)
    print(f"{n:>5} envs: {graph:>9,.0f} env-steps/s with the CUDA graph | {eager:>9,.0f} without | x{graph / eager:.1f}")

   64 envs:     8,956 env-steps/s with the CUDA graph |       515 without | x17.4
 1024 envs:   113,698 env-steps/s with the CUDA graph |     6,305 without | x18.0
 4096 envs:   371,592 env-steps/s with the CUDA graph |    21,165 without | x17.6


In [6]:
env.close()